# Data manipulation and analysis with Pandas

> **Explain it like I am five:** Raw data is a messy box of puzzle pieces. Data manipulation means cleaning the pieces, sorting them, joining matching pieces, and arranging them so the picture becomes visible.

This notebook preserves the original missing-value, renaming, type conversion, grouping, and merge examples. It corrects the column-name mismatch and adds the transformations used in real analysis.

## Learning goals

- diagnose and handle missing values;
- rename columns and convert data types safely;
- filter, sort, transform, aggregate, and group;
- merge, concatenate, pivot, and reshape tables;
- work with dates and build clear method chains.


In [1]:
import numpy as np
import pandas as pd

print("Pandas version:", pd.__version__)


Pandas version: 3.0.3


In [2]:
from pathlib import Path

def data_path(filename):
    """Find a course data file whether Jupyter starts in the repo or lesson folder."""
    current = Path.cwd().resolve()
    lesson_parts = ("Complete-Python-Bootcamp-main", "10-Data Analysis With Python")
    candidates = [current / filename, current.joinpath(*lesson_parts, filename)]
    for parent in current.parents:
        candidates.extend([parent / filename, parent.joinpath(*lesson_parts, filename)])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r}. Start Jupyter inside this repository.")


## 1. Load and inspect the data

Never clean a table blindly. First inspect its size, columns, types, sample rows, and missing values.


In [3]:
df=pd.read_csv(data_path('data.csv'))
## fetch the first 5 rows
df.head(5)


,Date,Category,Value,Product,Sales,Region
0,2023-01-01,A,28.0,Product1,754.0,East
1,2023-01-02,B,39.0,Product3,110.0,North
2,2023-01-03,C,32.0,Product2,398.0,East
3,2023-01-04,B,8.0,Product1,522.0,East
4,2023-01-05,B,26.0,Product3,869.0,North


In [4]:
print("last rows:\n", df.tail(5))
print("shape:", df.shape)
print("dtypes:\n", df.dtypes)


last rows:
           Date Category  Value   Product  Sales Region
45  2023-02-15        B   99.0  Product2  599.0   West
46  2023-02-16        B    6.0  Product1  938.0  South
47  2023-02-17        B   69.0  Product3  143.0   West
48  2023-02-18        C   65.0  Product3  182.0  North
49  2023-02-19        C   11.0  Product3  708.0  North
shape: (50, 6)
dtypes:
 Date            str
Category        str
Value       float64
Product         str
Sales       float64
Region          str
dtype: object


In [5]:
df.describe(include='all')


,Date,Category,Value,Product,Sales,Region
count,50,50,47.000000,50,46.000000,50
unique,50,3,NaN,3,NaN,4
top,2023-01-01,C,NaN,Product3,NaN,West
freq,1,21,NaN,19,NaN,18
mean,NaN,NaN,51.744681,NaN,557.130435,NaN
std,NaN,NaN,29.050532,NaN,274.598584,NaN
min,NaN,NaN,2.000000,NaN,108.000000,NaN
25%,NaN,NaN,27.500000,NaN,339.000000,NaN
50%,NaN,NaN,54.000000,NaN,591.500000,NaN
75%,NaN,NaN,70.000000,NaN,767.500000,NaN


`describe()` does not replace inspection. It summarizes distributions, while `head()` shows the actual arrangement and `dtypes` reveals how Pandas currently understands each column.


## 2. Find missing values

Missing values are like empty answer boxes. Before filling them, ask why they are missing and whether filling would distort the meaning.


In [6]:
## Handling Missing Values
print(df.isnull().any())


Date        False
Category    False
Value        True
Product     False
Sales        True
Region      False
dtype: bool


In [7]:
print(df.isnull().sum())


Date        0
Category    0
Value       3
Product     0
Sales       4
Region      0
dtype: int64


In [8]:
missing_report = (
    df.isna().sum()
    .rename('Missing_Count')
    .to_frame()
    .assign(Missing_Percent=lambda table: table['Missing_Count'] / len(df) * 100)
    .sort_values('Missing_Count', ascending=False)
)
missing_report


,Missing_Count,Missing_Percent
Sales,4,8.0
Value,3,6.0
Category,0,0.0
Date,0,0.0
Product,0,0.0
Region,0,0.0


## 3. Fill, drop, or preserve missing values

Common choices:

- fill with a constant when the constant has a real meaning;
- fill numeric values with median/mean when justified;
- fill within groups when groups have different typical values;
- drop rows only when losing them is acceptable;
- preserve missingness when “unknown” is itself useful information.


In [9]:
df_filled=df.fillna(0)
print(df_filled.head())


         Date Category  Value   Product  Sales Region
0  2023-01-01        A   28.0  Product1  754.0   East
1  2023-01-02        B   39.0  Product3  110.0  North
2  2023-01-03        C   32.0  Product2  398.0   East
3  2023-01-04        B    8.0  Product1  522.0   East
4  2023-01-05        B   26.0  Product3  869.0  North


Filling every column with zero is kept from the original lesson, but it is rarely the best real-world default. Zero sales may mean “none,” while a missing value may mean “not recorded.” Those are different stories.


In [10]:
### filling missing values with the mean of the column
df['Sales_fillNA']=df['Sales'].fillna(df['Sales'].mean())
df.head()


,Date,Category,Value,Product,Sales,Region,Sales_fillNA
0,2023-01-01,A,28.0,Product1,754.0,East,754.0
1,2023-01-02,B,39.0,Product3,110.0,North,110.0
2,2023-01-03,C,32.0,Product2,398.0,East,398.0
3,2023-01-04,B,8.0,Product1,522.0,East,522.0
4,2023-01-05,B,26.0,Product3,869.0,North,869.0


In [11]:
# Median is less affected by unusually large values.
df['Value_filled_median'] = df['Value'].fillna(df['Value'].median())

# Group-aware filling: use each product's typical sales first.
product_sales_median = df.groupby('Product')['Sales'].transform('median')
df['Sales_filled_by_product'] = df['Sales'].fillna(product_sales_median)

df[['Product', 'Sales', 'Sales_fillNA', 'Sales_filled_by_product']].head(10)


,Product,Sales,Sales_fillNA,Sales_filled_by_product
0,Product1,754.0,754.0,754.0
1,Product3,110.0,110.0,110.0
2,Product2,398.0,398.0,398.0
3,Product1,522.0,522.0,522.0
4,Product3,869.0,869.0,869.0
5,Product3,192.0,192.0,192.0
6,Product1,936.0,936.0,936.0
7,Product1,488.0,488.0,488.0
8,Product3,772.0,772.0,772.0
9,Product2,834.0,834.0,834.0


In [12]:
complete_rows = df.dropna(subset=['Value', 'Sales'])
print("original rows:", len(df))
print("complete rows:", len(complete_rows))


original rows: 50
complete rows: 43


## 4. Rename columns correctly

The original code tried to rename `'Sale Date'`, but the CSV column is named `'Date'`. Pandas silently ignores a missing rename key, which can hide mistakes. Check columns before and after.


In [13]:
## Renaming Columns
df=df.rename(columns={'Date':'Sales Date'})
print(df.columns.tolist())
df.head()


['Sales Date', 'Category', 'Value', 'Product', 'Sales', 'Region', 'Sales_fillNA', 'Value_filled_median', 'Sales_filled_by_product']


,Sales Date,Category,Value,Product,Sales,Region,Sales_fillNA,Value_filled_median,Sales_filled_by_product
0,2023-01-01,A,28.0,Product1,754.0,East,754.0,28.0,754.0
1,2023-01-02,B,39.0,Product3,110.0,North,110.0,39.0,110.0
2,2023-01-03,C,32.0,Product2,398.0,East,398.0,32.0,398.0
3,2023-01-04,B,8.0,Product1,522.0,East,522.0,8.0,522.0
4,2023-01-05,B,26.0,Product3,869.0,North,869.0,26.0,869.0


For machine-friendly columns, a consistent cleaning rule can remove spaces and standardize case.


In [14]:
machine_df = df.rename(columns=lambda name: name.strip().lower().replace(' ', '_'))
print(machine_df.columns.tolist())


['sales_date', 'category', 'value', 'product', 'sales', 'region', 'sales_fillna', 'value_filled_median', 'sales_filled_by_product']


## 5. Change data types safely

Converting a column can fail if text or missing values are present. `pd.to_numeric(..., errors='coerce')` changes invalid values to missing so they can be inspected explicitly.


In [15]:
## change datatypes
df['Value_new']=df['Value'].fillna(df['Value'].mean()).round().astype(int)
print(df[['Value', 'Value_new']].head())
print(df.dtypes)


   Value  Value_new
0   28.0         28
1   39.0         39
2   32.0         32
3    8.0          8
4   26.0         26
Sales Date                     str
Category                       str
Value                      float64
Product                        str
Sales                      float64
Region                         str
Sales_fillNA               float64
Value_filled_median        float64
Sales_filled_by_product    float64
Value_new                    int64
dtype: object


In [16]:
df['Sales Date'] = pd.to_datetime(df['Sales Date'], errors='coerce')
df['Category'] = df['Category'].astype('category')
print(df.dtypes)


Sales Date                 datetime64[us]
Category                         category
Value                             float64
Product                               str
Sales                             float64
Region                                str
Sales_fillNA                      float64
Value_filled_median               float64
Sales_filled_by_product           float64
Value_new                           int64
dtype: object


## 6. Transform columns

Vectorized arithmetic is usually clearer and faster than `apply` for simple math. The original `apply` example remains useful when the rule is a Python function.


In [17]:
df['New Value']=df['Value'].apply(lambda x:x*2)
df[['Value', 'New Value']].head()


,Value,New Value
0,28.0,56.0
1,39.0,78.0
2,32.0,64.0
3,8.0,16.0
4,26.0,52.0


In [18]:
df['New Value vectorized'] = df['Value'] * 2
print(df[['Value', 'New Value', 'New Value vectorized']].head())


   Value  New Value  New Value vectorized
0   28.0       56.0                  56.0
1   39.0       78.0                  78.0
2   32.0       64.0                  64.0
3    8.0       16.0                  16.0
4   26.0       52.0                  52.0


### `map`, `apply`, and vectorization

- use vectorized operations for arithmetic and string/date accessors;
- use `Series.map` for one value at a time or dictionary mapping;
- use `DataFrame.apply` for a row/column rule that cannot be expressed directly;
- avoid row-wise `apply` when a built-in vectorized operation exists.


In [19]:
region_group = {'East': 'E', 'West': 'W', 'North': 'N', 'South': 'S'}
df['Region Code'] = df['Region'].map(region_group)
df[['Region', 'Region Code']].drop_duplicates().sort_values('Region')


,Region,Region Code
0,East,E
1,North,N
12,South,S
5,West,W


## 7. Filter, query, and sort


In [20]:
filtered = df.loc[
    (df['Region'].isin(['East', 'West'])) & (df['Value_new'] >= 30),
    ['Sales Date', 'Product', 'Region', 'Value_new', 'Sales_fillNA']
].sort_values(['Region', 'Value_new'], ascending=[True, False])

filtered.head(10)


,Sales Date,Product,Region,Value_new,Sales_fillNA
41,2023-02-11,Product1,East,97,256.0
44,2023-02-14,Product3,East,96,830.0
13,2023-01-14,Product1,East,69,423.0
19,2023-01-20,Product1,East,59,736.0
43,2023-02-13,Product3,East,43,949.0
36,2023-02-06,Product3,East,36,177.0
2,2023-01-03,Product2,East,32,398.0
45,2023-02-15,Product2,West,99,599.0
25,2023-01-26,Product1,West,95,584.0
42,2023-02-12,Product3,West,93,164.0


In [21]:
# query is readable when column names are identifier-friendly.
machine_df = machine_df.assign(value_new=machine_df['value'].fillna(machine_df['value'].mean()))
machine_df.query("region == 'East' and value_new >= 30").head()


,sales_date,category,value,product,sales,region,sales_fillna,value_filled_median,sales_filled_by_product,value_new
2,2023-01-03,C,32.0,Product2,398.0,East,398.0,32.0,398.0,32.0
13,2023-01-14,A,69.0,Product1,423.0,East,423.0,69.0,423.0,69.0
19,2023-01-20,A,59.0,Product1,736.0,East,736.0,59.0,736.0,59.0
36,2023-02-06,C,36.0,Product3,177.0,East,177.0,36.0,177.0,36.0
41,2023-02-11,C,97.0,Product1,256.0,East,256.0,97.0,256.0,97.0


## 8. Aggregate and group

**Split → apply → combine:** `groupby` splits rows into labeled baskets, applies a calculation, then combines the answers.


In [22]:
## Data Aggregating And Grouping
grouped_mean=df.groupby('Product', observed=True)['Value'].mean()
print(grouped_mean)


Product
Product1    46.214286
Product2    52.800000
Product3    55.166667
Name: Value, dtype: float64


In [23]:
grouped_sum=df.groupby(['Product','Region'], observed=True)['Value'].sum()
print(grouped_sum)


Product   Region
Product1  East      292.0
          North       9.0
          South     100.0
          West      246.0
Product2  East       56.0
          North     127.0
          South     181.0
          West      428.0
Product3  East      202.0
          North     203.0
          South     215.0
          West      373.0
Name: Value, dtype: float64

In [24]:
df.groupby(['Product','Region'], observed=True)['Value'].mean()


Product   Region
Product1  East      41.714286
          North      4.500000
          South     50.000000
          West      82.000000
Product2  East      28.000000
          North     63.500000
          South     60.333333
          West      53.500000
Product3  East      50.500000
          North     40.600000
          South     71.666667
          West      62.166667
Name: Value, dtype: float64

In [25]:
## aggregate multiple functions
grouped_agg=df.groupby('Region', observed=True)['Value'].agg(['mean','sum','count'])
grouped_agg


,mean,sum,count
Region,,,
East,42.307692,550.0,13
North,37.666667,339.0,9
South,62.000000,496.0,8
West,61.588235,1047.0,17


Named aggregation creates understandable output column names.


In [26]:
region_summary = (
    df.groupby('Region', as_index=False, observed=True)
    .agg(
        Average_Value=('Value', 'mean'),
        Total_Sales=('Sales', 'sum'),
        Recorded_Sales=('Sales', 'count'),
        Products=('Product', 'nunique')
    )
    .sort_values('Total_Sales', ascending=False)
)
region_summary


,Region,Average_Value,Total_Sales,Recorded_Sales,Products
3,West,61.588235,7445.0,16,3
0,East,42.307692,7017.0,12,3
1,North,37.666667,6008.0,10,3
2,South,62.000000,5158.0,8,3


## 9. `agg` vs. `transform` vs. `filter`

- `agg` makes fewer summary rows;
- `transform` returns one aligned value per original row;
- `filter` keeps or removes whole groups.


In [27]:
df['Product Mean Value'] = df.groupby('Product', observed=True)['Value'].transform('mean')
df['Difference From Product Mean'] = df['Value'] - df['Product Mean Value']

large_groups = df.groupby('Product', observed=True).filter(lambda group: len(group) >= 10)
print(df[['Product', 'Value', 'Product Mean Value', 'Difference From Product Mean']].head())
print("rows in groups with at least 10 records:", len(large_groups))


    Product  Value  Product Mean Value  Difference From Product Mean
0  Product1   28.0           46.214286                    -18.214286
1  Product3   39.0           55.166667                    -16.166667
2  Product2   32.0           52.800000                    -20.800000
3  Product1    8.0           46.214286                    -38.214286
4  Product3   26.0           55.166667                    -29.166667
rows in groups with at least 10 records: 50


## 10. Merge DataFrames

A merge matches rows using key values. Imagine two sticker books that both use the same student ID.

- inner: keys present on both sides;
- left: every left key plus matches;
- right: every right key plus matches;
- outer: every key from either side.


In [28]:
### Merging and joining Dataframes
# Create sample DataFrames
df1 = pd.DataFrame({'Key': ['A', 'B', 'C'], 'Value1': [1, 2, 3]})
df2 = pd.DataFrame({'Key': ['A', 'B', 'D'], 'Value2': [4, 5, 6]})
print(df1)
print(df2)


  Key  Value1
0   A       1
1   B       2
2   C       3
  Key  Value2
0   A       4
1   B       5
2   D       6


In [29]:
print("inner:\n", pd.merge(df1,df2,on="Key",how="inner"))
print("outer:\n", pd.merge(df1,df2,on="Key",how="outer"))
print("left:\n", pd.merge(df1,df2,on="Key",how="left"))
print("right:\n", pd.merge(df1,df2,on="Key",how="right"))


inner:
   Key  Value1  Value2
0   A       1       4
1   B       2       5
outer:
   Key  Value1  Value2
0   A     1.0     4.0
1   B     2.0     5.0
2   C     3.0     NaN
3   D     NaN     6.0
left:
   Key  Value1  Value2
0   A       1     4.0
1   B       2     5.0
2   C       3     NaN
right:
   Key  Value1  Value2
0   A     1.0       4
1   B     2.0       5
2   D     NaN       6


Use `validate` to state the expected key relationship and `indicator=True` to audit matches.


In [30]:
audited_merge = pd.merge(
    df1,
    df2,
    on='Key',
    how='outer',
    validate='one_to_one',
    indicator=True
)
audited_merge


,Key,Value1,Value2,_merge
0,A,1.0,4.0,both
1,B,2.0,5.0,both
2,C,3.0,NaN,left_only
3,D,NaN,6.0,right_only


## 11. Concatenate tables

`concat` stacks tables by rows (`axis=0`) or places them side by side (`axis=1`). It does not match database keys like `merge`.


In [31]:
first_half = df.iloc[:3][['Product', 'Region']]
second_half = df.iloc[3:6][['Product', 'Region']]
stacked = pd.concat([first_half, second_half], ignore_index=True)
print(stacked)


    Product Region
0  Product1   East
1  Product3  North
2  Product2   East
3  Product1   East
4  Product3  North
5  Product3   West


## 12. Pivot wider and melt longer

`pivot_table` turns category values into columns and summarizes duplicates. `melt` reverses wide columns into name/value rows.


In [32]:
pivot = pd.pivot_table(
    df,
    index='Product',
    columns='Region',
    values='Sales',
    aggfunc='sum',
    fill_value=0,
    observed=True
)
print(pivot)

long_again = pivot.reset_index().melt(
    id_vars='Product',
    var_name='Region',
    value_name='Sales'
)
print(long_again.head())


Region      East   North   South    West
Product                                 
Product1  4205.0  1737.0  1346.0  1335.0
Product2   856.0   843.0  2240.0  3435.0
Product3  1956.0  3428.0  1572.0  2675.0
    Product Region   Sales
0  Product1   East  4205.0
1  Product2   East   856.0
2  Product3   East  1956.0
3  Product1  North  1737.0
4  Product2  North   843.0


## 13. Time-based analysis

Once dates have a datetime dtype, Pandas can extract calendar parts, resample periods, and calculate rolling windows.


In [33]:
daily_sales = (
    df.set_index('Sales Date')['Sales']
    .sort_index()
    .resample('7D')
    .sum(min_count=1)
)
rolling_average = daily_sales.rolling(window=3, min_periods=1).mean()

print(pd.DataFrame({'Sales': daily_sales, 'Rolling_3_Period': rolling_average}).head())


             Sales  Rolling_3_Period
Sales Date                          
2023-01-01  3781.0       3781.000000
2023-01-08  3987.0       3884.000000
2023-01-15  4327.0       4031.666667
2023-01-22  4161.0       4158.333333
2023-01-29  3285.0       3924.333333


## 14. A readable method chain

A method chain reads like a recipe. Give complex steps meaningful names or helper functions instead of making one enormous chain.


In [34]:
analysis_ready = (
    pd.read_csv(data_path('data.csv'))
    .rename(columns={'Date': 'Sales Date'})
    .assign(
        **{
            'Sales Date': lambda table: pd.to_datetime(table['Sales Date']),
            'Value': lambda table: table['Value'].fillna(table['Value'].median()),
            'Sales': lambda table: table['Sales'].fillna(table['Sales'].median())
        }
    )
    .sort_values('Sales Date')
    .reset_index(drop=True)
)
analysis_ready.head()


,Sales Date,Category,Value,Product,Sales,Region
0,2023-01-01,A,28.0,Product1,754.0,East
1,2023-01-02,B,39.0,Product3,110.0,North
2,2023-01-03,C,32.0,Product2,398.0,East
3,2023-01-04,B,8.0,Product1,522.0,East
4,2023-01-05,B,26.0,Product3,869.0,North


## 15. Common mistakes

- filling every missing value with zero without considering meaning;
- converting to integer before handling missing or invalid values;
- renaming a nonexistent column and assuming it worked;
- using `apply` for simple vectorized arithmetic;
- confusing `agg` (smaller output) with `transform` (same row count);
- merging without checking whether keys are unique;
- forgetting that `pivot` fails on duplicate index/column pairs while `pivot_table` aggregates them;
- treating date strings as calendar-aware values.


## 16. Mini practice

1. Fill missing `Sales` with each region's median.
2. Calculate each row's percentage of its product's total sales.
3. Make a region-by-product sales pivot table.
4. Audit an outer merge with `indicator=True`.


In [35]:
practice = pd.read_csv(data_path('data.csv'))
region_median = practice.groupby('Region')['Sales'].transform('median')
practice['Sales'] = practice['Sales'].fillna(region_median)
product_total = practice.groupby('Product')['Sales'].transform('sum')
practice['Product_Sales_Share'] = practice['Sales'] / product_total

print(practice[['Product', 'Region', 'Sales', 'Product_Sales_Share']].head())
print(pd.pivot_table(
    practice,
    index='Region',
    columns='Product',
    values='Sales',
    aggfunc='sum',
    fill_value=0
))


    Product Region  Sales  Product_Sales_Share
0  Product1   East  754.0             0.081193
1  Product3  North  110.0             0.010804
2  Product2   East  398.0             0.047156
3  Product1   East  522.0             0.056211
4  Product3  North  869.0             0.085355


Product  Product1  Product2  Product3
Region                               
East       4205.0     856.0    2506.0
North      2400.5     843.0    3428.0
South      1346.0    2240.0    1572.0
West       1335.0    4501.0    2675.0


## Easy revision cheat sheet

| Goal | Pattern |
|---|---|
| Count missing | `df.isna().sum()` |
| Fill | `s.fillna(value)` |
| Drop incomplete rows | `df.dropna(subset=[...])` |
| Rename | `df.rename(columns={'old': 'new'})` |
| Convert numeric | `pd.to_numeric(s, errors='coerce')` |
| Convert date | `pd.to_datetime(s, errors='coerce')` |
| Filter | `df.loc[condition, columns]` |
| Map labels | `s.map(dictionary)` |
| Group summary | `df.groupby(key).agg(...)` |
| Group-aligned result | `df.groupby(key)[col].transform(...)` |
| Join by key | `pd.merge(left, right, on=key, how=...)` |
| Audit join | `validate=...`, `indicator=True` |
| Stack tables | `pd.concat([...], ignore_index=True)` |
| Long to wide | `pd.pivot_table(...)` |
| Wide to long | `pd.melt(...)` or `df.melt(...)` |
| Time buckets | `series.resample('7D').sum()` |
| Moving window | `series.rolling(window).mean()` |

**Memory trick:** Cleaning fixes pieces, `groupby` sorts pieces into baskets, `merge` connects matching labels, and pivot changes the table's shape.
